## How do models perform pragmatic inference?

In this notebook, we'll walk through a toy example of pragmatic inference. We'll see how a rational listener (under the Rational Speech Acts, or RSA, model) would make the pragmatic inference. Then, we'll try to see whether language models *behave* the same way, and - hopefully - whether they *represent* things the same way in their underlying computation.

## 0️⃣ Setup

Run this code to get started!

In [ ]:
from IPython.display import clear_output
import plotly.io as pio

try:
    import google.colab
    is_colab = True
except ImportError:
    is_colab = False

if is_colab:
    pio.renderers.default = "colab"
    !git clone https://github.com/rkapur102/cs221m-course.git
    %cd cs221m-course
    !pip install -r requirements/extra_rsa_pragmatics.txt
else:
    pio.renderers.default = "plotly_mimetype+png"
    !plotly_get_chrome -y

clear_output()

### Pragmatic inference

For our running example, imagine a reference game where the speaker says a single word, and the listener has to guess the object that they're talking about.

Imagine that the objects in front of us are a blue square, a blue circle, and a green square.

<img src="https://raw.githubusercontent.com/rkapur102/cs221m-course/refs/heads/main/figures/extra_rsa_pragmatics/reference_game.png" width=40%>

In theory, the speaker can utter any single word (e.g., "sky") but for simplicity, let's assume that they can only use one of the following: "square", "circle", "blue", and "green". This means that with their utterance, the speaker can only pick out a single feature of the objects (either the shape or the color) but not both.

Let's pretend to play the game. Imagine the speaker says "circle" - which object should we pick? Hopefully you chose the blue circle without too much trouble. After all, there's only one circle. Similarly, if the speaker says "green", then we would pick out the green square, since it's the only green object.

What if the speaker says "blue"? Naively, we might pick either the blue square or the blue circle with a fifty-fifty chance, since "blue" suits them both equally. But we can do better if we put ourselves in the speaker's shoes. We just saw that if the speaker wanted us to choose the blue circle, they could've said "circle", and there would've been no confusion about what to pick. However, they can't do the same with the blue square: "square" refers to two different objects (both the blue and the green square). This suggests that the speaker likely didn't mean for us to choose the blue circle - otherwise they would've just said "circle"! - and so, pragmatically, "blue" is more likely to refer to the blue square than the blue circle (even though *semantically* it refers to both equally!). 

### Rational Speech Acts (RSA)

The RSA model seeks to turn the intuition above (putting ourselves "in the speaker's shoes") into a computational model of how people infer the meaning of a sentence in different context. For a more in-depth introduction, see the [RSA textbook](https://www.problang.org/) by Scontras, Tessler, and Franke.

<img src="https://raw.githubusercontent.com/rkapur102/cs221m-course/refs/heads/main/figures/extra_rsa_pragmatics/rsa.png" width=50%>

In RSA, the listener recursively reasons about the speaker, who reasons about the listener:
 - the **pragmatic listener** (**L1**) picks referents based on what the **pragmatic speaker** (**S1**) would say
 - the **pragmatic speaker** (**S1**) in turn chooses utterances based on what they think the **literal listener** (**L0**) would pick
 - the **literal listener** (**L0**) chooses referents from the literal meaning of the utterance

We'll go through these one at a time, starting with the literal listener.

**Literal Listener (L0)**: the literal listener chooses the objects proportionately to the meaning of the utterance. Since "blue" is true for both the blue square and the blue circle, the literal listener will choose these objects with equal probability.

$P_{L_0}(r \mid u) \propto \llbracket u \rrbracket(r)$

In [1]:
import numpy as np

REFERENCES = ["blue square", "blue circle", "green square"] # R
UTTERANCES = ["blue", "green", "square", "circle"] # U

MEANINGS = np.array([
    [1, 1, 0], # blue
    [0, 0, 1], # green
    [1, 0, 1], # square
    [0, 1, 0]  # circle
])

def literal_listener(u : int):
    """
    L0: P(r | u)
    literal meaning (i.e., "blue" is equally likely to mean blue square & blue circle)
    """
    # the literal listener is proportional to the literal meanings
    # i.e., we divide each row by its sum (e.g., blue [1, 1, 0] -> blue [0.5, 0.5, 0])
    L0 = MEANINGS / MEANINGS.sum(axis=1)[:, None]
    return L0[u]

# play around with the utterance!
utterance = "blue"

u = UTTERANCES.index(utterance) # convert to integer
l0 = literal_listener(u)

print("Literal listener")
print(f"Heard: {utterance}")
print("\t|\t".join(REFERENCES))
print("\t\t|\t".join([f"{p:.2f}" for p in l0]))

Literal listener
Heard: blue
blue square	|	blue circle	|	green square
0.50		|	0.50		|	0.00


**Pragmatic speaker (S1)**: the pragmatic speaker chooses the utterance proportionately to the literal listener. That is, if the speaker wants to talk about the blue circle, they will be more likely to say "circle" than "blue", because the literal listener assigns a higher probability to blue circle upon hearing "circle" (100%) than upon hearing "blue" (50%). Note: this is a slight simplification of the RSA model published by Frank & Goodman (2012), but since we're focusing on a toy task, it's okay. 

$P_{S_1}(u \mid r) \propto P_{L_0}(r \mid u)$

In [2]:
def pragmatic_speaker(r : int):
    """
    S1: P(u | r)
    what would the speaker choose to refer to r?
    """
    # literal listener: proportional to the actual meanings
    L0 = MEANINGS / MEANINGS.sum(axis=1)[:, None]
    # pragmatic speaker: proportional to the literal listener
    # i.e., we transpose L0 (flip rows & columns) and divide each row by its sum
    S1 = L0.T / L0.T.sum(axis=1)[:, None]
    return S1[r]

# play around with the referent!
referent = "blue circle"

r = REFERENCES.index(referent)
s1 = pragmatic_speaker(r)

print("Pragmatic speaker")
print(f"Want to refer to: {referent}")
print("\t|\t".join(UTTERANCES))
print("\t|\t".join([f"{p:.2f}" for p in s1]))

Pragmatic speaker
Want to refer to: blue circle
blue	|	green	|	square	|	circle
0.33	|	0.00	|	0.00	|	0.67


**Pragmatic listener (L1)**: the pragmatic listener chooses the referent proportionately to the pragmatic speaker. That is, since the speaker is more likely to say "blue" when talking about the blue square than the blue circle (50% vs. 33%), the pragmatic listener will assign greater probability to the blue square than the circle upon hearing "blue".

$P_{L_1}(r \mid u) \propto P_{S_1}(u \mid r)$

In [3]:
def pragmatic_listener(u : int):
    """
    L1: P(r | u)
    what did the speaker pragmatically refer to?
    """
    # literal listener: proportional to actual meaning
    L0 = MEANINGS / MEANINGS.sum(axis=1)[:, None]
    # pragmatic speaker: proportional to literal listener
    S1 = L0.T / L0.T.sum(axis=1)[:, None]
    # pragmatic listener: proportional to pragmatic speaker
    # i.e., we transpose S1 (flip rows & columns) and divide each row by its sum
    L1 = S1.T / S1.T.sum(axis=1)[:, None]
    return L1[u]

# play around with the utterance!
utterance = "blue"

u = UTTERANCES.index(utterance) # convert to integer
l0 = literal_listener(u)
l1 = pragmatic_listener(u)


print(f"Heard: {utterance}")
print()
print("Literal listener")
print("\t|\t".join(REFERENCES))
print("\t\t|\t".join([f"{p:.2f}" for p in l0]))
print()
print("Pragmatic listener")
print("\t|\t".join(REFERENCES))
print("\t\t|\t".join([f"{p:.2f}" for p in l1]))

Heard: blue

Literal listener
blue square	|	blue circle	|	green square
0.50		|	0.50		|	0.00

Pragmatic listener
blue square	|	blue circle	|	green square
0.60		|	0.40		|	0.00


And there we have it! Under the RSA model, listeners pragmatically infer what the speaker is talking about by reasoning about how the speaker chose what to say. In the case of ambiguous statements like "blue" when there's a blue square and a blue circle on the table, the listener rules out referents like blue circle that the speaker could've referred to using a different utterance (i.e., "circle").

The RSA model is a flexible computational model that has shown success in modeling human behavior. Can it also account for how language models make pragmatic inferences? Better yet, can it explain how language models *represent* listener/speaker beliefs in their underlying computation?

Let's test if language models behave like pragmatic listeners!

We'll use NDIF to run experiments on large language models without having to load them ourselves. To get an API key, log in with your Stanford account here: https://login.api.ndif.us/.

In [1]:
import nnsight
nnsight.login()

NDIF API key saved — logged in as amirzur@stanford.edu.


We'll use Qwen3.5-27B, hosted on NDIF. I'd recommend playing around with smaller models as long as they answer the prompts correctly. But as we'll see, even a 27B model doesn't act all that pragmatically!

In [5]:
model = nnsight.TransformersModel("Qwen/Qwen3.5-27B")
tokenizer = model.tokenizer

W1117 15:27:44.400000 86821 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


I'd definitely recommend playing around with the prompt template! There's a small figure to help visualize what's currently in the prompt.

### What about language models?

In [6]:
NAMES = ["Gary", "Charles", "Frank"]
token_ids = [ids[1] if len(ids) > 1 else ids[0] for ids in tokenizer(NAMES).input_ids] # convert to token ids

PROMPT_TEMPLATE = """Alice and Bob are playing a collaborative game where Alice says a word and Bob has to pick the thing that Alice is referring to.

In front of Bob are a blue dog called Gary, a green dog called Frank, and a blue cat called Charles. 

Alice sees the animals, but doesn't know their names. Instead, she can only say \"blue\", \"green\", \"dog\", or \"cat\".

Alice says, \"{utterance}\".

What animal is Alice referring to?
<think></think>
Answer: the animal that Alice is referring to is called **"""

def get_prompt(utterance : str):
    return PROMPT_TEMPLATE.format(utterance=utterance)

<img src="https://raw.githubusercontent.com/rkapur102/cs221m-course/refs/heads/main/figures/extra_rsa_pragmatics/reference_game_lm.png" width=40%>

In [7]:
print(get_prompt(utterance="dog"))

Alice and Bob are playing a collaborative game where Alice says a word and Bob has to pick the thing that Alice is referring to.

In front of Bob are a blue dog called Gary, a green dog called Frank, and a blue cat called Charles. 

Alice sees the animals, but doesn't know their names. Instead, she can only say "blue", "green", "dog", or "cat".

Alice says, "dog".

What animal is Alice referring to?
<think></think>
Answer: the animal that Alice is referring to is called **


If Alice says "dog", is she more likely to be referring to Gary or Frank? (For which one could she have chosen a simpler utterance?)

In [8]:
utterance = "dog"

prompt = get_prompt(utterance)
with model.session(remote=True):
    with model.trace(prompt):
        # get logits over last token in prompt
        logits = model.output.logits[:, -1, :].save()

Let's see if our language model thinks the same way...

In [9]:
import torch

probs = torch.softmax(logits, dim=-1)

print(f"Heard: {utterance}")
print("\t|\t".join(n.strip() for n in NAMES))
print("\t|\t".join(f"{p:.2f}" for p in probs[0, token_ids]))

Heard: dog
Gary	|	Charles	|	Frank
0.46	|	0.01	|	0.40


Why don't these add up to 1? The softmax runs over the model's entire vocabulary, so some probability goes to tokens other than our three names (a different name, a space, "the", and so on). What we print is only the part of that distribution that lands on Gary, Charles, and Frank.

If we only care about how the model splits its belief among the three animals, we can renormalize by dividing each probability by their sum:

In [10]:
name_probs = probs[0, token_ids]
print(f"Total probability on the three names: {name_probs.sum():.2f}")
print()
print("Renormalized")
print("\t|\t".join(n.strip() for n in NAMES))
print("\t|\t".join(f"{p:.2f}" for p in name_probs / name_probs.sum()))

Total probability on the three names: 0.87

Renormalized
Gary	|	Charles	|	Frank
0.52	|	0.01	|	0.46


It's slim, but it looks like Qwen figured out that "dog" is more likely to refer to Gary than Frank!

<img src="https://raw.githubusercontent.com/rkapur102/cs221m-course/refs/heads/main/figures/extra_rsa_pragmatics/reference_game_lm.png" width=40%>

Let's try different utterances!

In [11]:
utterance = "cat"

prompt = get_prompt(utterance)
with model.session(remote=True):
    with model.trace(prompt):
        # get logits over last token in prompt
        logits = model.output.logits[:, -1, :].save()

probs = torch.softmax(logits, dim=-1)

print(f"Heard: {utterance}")
print("\t|\t".join(n.strip() for n in NAMES))
print("\t|\t".join(f"{p:.2f}" for p in probs[0, token_ids]))

Heard: cat
Gary	|	Charles	|	Frank
0.00	|	0.99	|	0.00


Seems pretty good! Perhaps it's not consistent, but it's in the right direction.

In [12]:
utterance = "blue"

prompt = get_prompt(utterance)
with model.session(remote=True):
    with model.trace(prompt):
        # get logits over last token in prompt
        logits = model.output.logits[:, -1, :].save()

probs = torch.softmax(logits, dim=-1)

print(f"Heard: {utterance}")
print("\t|\t".join(n.strip() for n in NAMES))
print("\t|\t".join(f"{p:.2f}" for p in probs[0, token_ids]))

Heard: blue
Gary	|	Charles	|	Frank
0.61	|	0.20	|	0.09


That said, these are just a few data points. It's hard to know whether Qwen is actually making pragmatic inferences, or whether it just has a bias towards the name Gary (or dogs, or the color blue!).

### Causal abstraction

In [13]:
NAMES = ["Gary", "Frank", "Charles"]
OBJECTS = ["blue dog", "green dog", "blue cat"]
UTTERANCES = ["blue", "green", "dog", "cat"]

PROMPT_TEMPLATE = """Alice and Bob are playing a collaborative game where Alice says a word and Bob has to pick the thing that Alice is referring to.

In front of Bob are {referents}. 

Alice sees the animals, but doesn't know their names. Instead, she can only say {utterances}.

Alice says, \"{utterance}\".

What animal is Alice referring to?
<think></think>
Answer: the animal that Alice is most likely referring to is the one called **"""

def get_prompt(utterance : str, 
               names : list[str] = NAMES, 
               objects : list[str] = OBJECTS, 
               utterances : list[str] = UTTERANCES):
    referents = [f"a {obj} called {name}" for obj, name in zip(objects, names)]
    referents[-1] = "and " + referents[-1]
    referents_str = ", ".join(referents)

    utterances = [f"\"{u}\"" for u in utterances]
    utterances[-1] = "or " + utterances[-1]
    utterances_str = ", ".join(utterances)
    
    return PROMPT_TEMPLATE.format(referents=referents_str, utterances=utterances_str, utterance=utterance)

def get_output_ids(names : list[str] = NAMES):
    # the answer is read off the token right after "**", so each name must be a single token
    # (e.g. Qwen3.5 splits "Eddie" into "E" + "ddie", and we would be reading the probability of "E")
    ids = tokenizer(names).input_ids
    assert all(len(i) == 1 for i in ids), f"pick names that are a single token: {names}"
    return [i[0] for i in ids]

In [14]:
prompt = get_prompt("dog")
print(prompt)

Alice and Bob are playing a collaborative game where Alice says a word and Bob has to pick the thing that Alice is referring to.

In front of Bob are a blue dog called Gary, a green dog called Frank, and a blue cat called Charles. 

Alice sees the animals, but doesn't know their names. Instead, she can only say "blue", "green", "dog", or "cat".

Alice says, "dog".

What animal is Alice referring to?
<think></think>
Answer: the animal that Alice is most likely referring to is the one called **


In [15]:
def print_output(prompt : str, names : list[str], output_ids : list[int]):
    with model.session(remote=True):
        with model.trace(prompt):
            # get logits over last token in prompt
            logits = model.output.logits[:, -1, :].save()

    probs = torch.softmax(logits, dim=-1)

    print("\t|\t".join(n.strip() for n in names))
    print("\t|\t".join(f"{p:.2f}" for p in probs[0, output_ids]))

<img src="https://raw.githubusercontent.com/rkapur102/cs221m-course/refs/heads/main/figures/extra_rsa_pragmatics/reference_game_lm.png" width=40%>

In [16]:
OBJECTS

['blue dog', 'green dog', 'blue cat']

In [17]:
names = ["Gary", "Tom", "David"]
output_ids = get_output_ids(names)

source_prompt = get_prompt("blue", objects=["green dog", "blue cat", "blue dog"], names=names)
target_prompt = get_prompt("blue", objects=["green cat", "blue cat", "blue dog"], names=names)

print("source")
print_output(source_prompt, names, output_ids)

print("target")
print_output(target_prompt, names, output_ids)

source


Gary	|	Tom	|	David
0.01	|	0.47	|	0.41
target


Gary	|	Tom	|	David
0.02	|	0.33	|	0.54


In [18]:
def interchange_intervention(source_prompt : str, target_prompt : str, token_indices : list[int], layers : list[int], output_ids : list[int]):
    # One trace per run (the same as interchange_intervention_slow in utils.py). A faster version runs all
    # layers as separate invokes of one trace, synced with barriers, but NDIF's multi-GPU deployment of
    # Qwen3.5-27B currently fails on multi-invoke traces once they reach later layers.
    counterfactual_outputs = []
    with model.session(remote=True):
        with torch.no_grad():
            # STEP 1: source run, saving the output of every layer we will patch
            with model.trace(source_prompt):
                hidden_states = {layer_idx: model.model.language_model.layers[layer_idx].output.save()
                                 for layer_idx in layers}

            # STEP 2: counterfactual runs (source -> target), one per layer and token position
            for layer_idx in layers:
                layer_counterfactual_outputs = []
                for token_idx in token_indices:
                    with model.trace(target_prompt):
                        # Patch (replace) the target's hidden state with the source's at this layer and position.
                        model.model.language_model.layers[layer_idx].output[:, token_idx, :] = hidden_states[layer_idx][:, token_idx, :]

                        counterfactual_logits = model.output.logits[:, -1, :]
                        result = nnsight.save(torch.softmax(counterfactual_logits, dim=-1)[0, output_ids].detach().cpu().tolist())
                    layer_counterfactual_outputs.append(result)
                counterfactual_outputs.append(layer_counterfactual_outputs)

            counterfactual_outputs = nnsight.save(counterfactual_outputs)

    return counterfactual_outputs

In [19]:
token_indices = [-1]
layers = list(range(0, len(model.model.language_model.layers), 5))

counterfactual_outputs = interchange_intervention(source_prompt, target_prompt, token_indices, layers, output_ids)

In [20]:
import pandas as pd
import plotly.express as px

df = pd.DataFrame([c[0] for c in counterfactual_outputs], columns=names, index=layers)
px.line(df, template="simple_white", width=600, markers=True,
        labels={"index": "Layer", "value": "P(name) after patching", "variable": "Name"})

In [21]:
names = ["Gary", "Tom", "David"]
objects = ["green dog", "blue cat", "blue dog"]
output_ids = get_output_ids(names)

source_prompt = get_prompt("blue", objects=objects, names=names)
target_prompt = get_prompt("dog", objects=objects, names=names)

print("source")
print_output(source_prompt, names, output_ids)

print("target")
print_output(target_prompt, names, output_ids)

source


Gary	|	Tom	|	David
0.01	|	0.47	|	0.41
target


Gary	|	Tom	|	David
0.27	|	0.04	|	0.64


In [22]:
print(target_prompt)

Alice and Bob are playing a collaborative game where Alice says a word and Bob has to pick the thing that Alice is referring to.

In front of Bob are a green dog called Gary, a blue cat called Tom, and a blue dog called David. 

Alice sees the animals, but doesn't know their names. Instead, she can only say "blue", "green", "dog", or "cat".

Alice says, "dog".

What animal is Alice referring to?
<think></think>
Answer: the animal that Alice is most likely referring to is the one called **


In [23]:
token_indices = [-1]
layers = list(range(0, len(model.model.language_model.layers), 5))

counterfactual_outputs_2 = interchange_intervention(source_prompt, target_prompt, token_indices, layers, output_ids)

In [24]:
import pandas as pd
import plotly.express as px

df_2 = pd.DataFrame([c[0] for c in counterfactual_outputs_2], columns=names, index=layers)
px.line(df_2, template="simple_white", width=600, markers=True,
        labels={"index": "Layer", "value": "P(name) after patching", "variable": "Name"})

### 📚 Readings

If you want to go deeper into anything in this notebook, we recommend the following.

* **Frank & Goodman** (2012), [*Predicting Pragmatic Reasoning in Language Games*](https://doi.org/10.1126/science.1218633). This introduces the model behind this notebook in a reference game much like ours, where a speaker names one of several objects with a single word. A listener who assumes the speaker chose the most informative word predicted people's choices closely.
* **Goodman & Frank** (2016), [*Pragmatic Language Interpretation as Probabilistic Inference*](https://doi.org/10.1016/j.tics.2016.08.005). This review presents RSA as a general framework in which listeners and speakers reason recursively about each other, and surveys how it extends to other pragmatic phenomena such as scalar implicature.
* **Scontras et al.** (2021), [*A Practical Introduction to the Rational Speech Act Modeling Framework*](https://arxiv.org/abs/2105.09867). This is the companion paper to the RSA textbook linked above. It reviews the framework's theoretical foundations, its technical developments, and its connections to other areas of linguistics and cognitive science.
* **Geiger et al.** (2021), [*Causal Abstractions of Neural Networks*](https://arxiv.org/abs/2106.02997). This turns causal abstraction into an analysis method for neural networks: align representations with the variables of an interpretable causal model, then use interchange interventions to check that they play the same causal roles. It is the basis for the interventions in the last section of this notebook.